# HRNet-W18 Jacquard V2 — Kaggle training

Run on a Kaggle notebook with a **T4** or **P100** accelerator (GPU > Settings > Accelerator). T4 is preferred; P100 has slightly less VRAM but works.

## One-time setup before opening this notebook on Kaggle

1. **Upload the dataset as a Kaggle Dataset** (this is the slow step; do it once).
   - On your local machine, download all 12 `JacquardV2_Dataset_*.zip` from your Yandex Disk folder.
   - Go to https://www.kaggle.com/datasets → **New Dataset** → drag-drop the 12 zips. Title: `Jacquard V2`. Slug example: `jacquard-v2-zips`.
   - Wait for processing (~30–60 min depending on your upload). Kaggle will keep the zips intact and accessible at `/kaggle/input/jacquard-v2-zips/`.
   - **Set the dataset to Private** if you don't want to publish it (the upstream Jacquard V2 license requires re-distribution under the same terms; check before making it Public).

2. **Open this notebook on Kaggle**: New Notebook → upload `kaggle_train.ipynb` → Settings panel → "Add Input" → search for your dataset slug → attach → Settings → Accelerator → GPU T4 ×1 → Save.

## What the notebook does
1. Clones the repo from GitHub into `/kaggle/working/`.
2. Installs `imagecodecs` (Kaggle preinstalls torch/timm/cv2/tifffile).
3. Unzips the 12 zips from `/kaggle/input/...` into `/kaggle/working/Jacquard_V2/` on the local SSD (~5 min, fast read from Kaggle's input cache).
4. Builds the object-wise split.
5. Trains. Checkpoints land in `/kaggle/working/runs/<run_name>/` and persist as the **notebook output**.
6. Auto-resumes if a previous run output is attached as another input.

## 0. User-supplied configuration

In [ ]:
# ====== EDIT THESE ======
KAGGLE_DATASET_DIR = "/kaggle/input/jacquard-v2-zips"  # change to match your dataset slug
RUN_NAME = "hrnet_w18_rgbd_angle"
GITHUB_REPO = "https://github.com/Hesgoryr/HRNet_Grasp_Semantic_Segmentation.git"
GITHUB_BRANCH = "main"
# Optional: attach a previous version of this notebook's output as an input dataset to auto-resume.
# Set RESUME_INPUT_DIR to the path where it was mounted (e.g. "/kaggle/input/hrnet-resume/runs/<RUN_NAME>").
RESUME_INPUT_DIR = None
# Training overrides (T4 has 16 GB VRAM)
IMAGE_SIZE = 384
BATCH_SIZE = 16
ACCUM_STEPS = 1
NUM_WORKERS = 4
EPOCHS = 80
# ========================

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone repo + install missing deps

Kaggle preinstalls torch/timm/cv2/tifffile/numpy. We only need `imagecodecs` (LZW depth TIFFs).

In [ ]:
import os
REPO_DIR = '/kaggle/working/HRNet_Grasp_Semantic_Segmentation'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -b {GITHUB_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth 1 origin {GITHUB_BRANCH} && git reset --hard origin/{GITHUB_BRANCH}
%cd {REPO_DIR}
!pip install -q imagecodecs

In [ ]:
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Unzip the input dataset to local SSD

Kaggle's `/kaggle/input/...` is read-only and slow for many small files; we unpack into `/kaggle/working/Jacquard_V2/` on the working volume (15 GB free typically — fits the unpacked dataset).

In [ ]:
import glob, zipfile, shutil

DATA_DIR = '/kaggle/working/Jacquard_V2'
os.makedirs(DATA_DIR, exist_ok=True)

zips = sorted(glob.glob(os.path.join(KAGGLE_DATASET_DIR, '*.zip')))
print(f'found {len(zips)} zips in {KAGGLE_DATASET_DIR}')
for z in zips:
    print('extracting', os.path.basename(z))
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)

# Flatten JacquardV2_Dataset_<N>/ shards under DATA_DIR.
for shard_dir in sorted(glob.glob(os.path.join(DATA_DIR, 'JacquardV2_Dataset_*'))):
    if not os.path.isdir(shard_dir):
        continue
    for entry in os.listdir(shard_dir):
        src = os.path.join(shard_dir, entry)
        dst = os.path.join(DATA_DIR, entry)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    try:
        os.rmdir(shard_dir)
    except OSError:
        pass

!find {DATA_DIR} -mindepth 1 -maxdepth 3 -name '*_grasps.txt' | wc -l

## 4. Build the object-wise split

In [ ]:
SPLITS_PATH = os.path.join(REPO_DIR, 'splits/jacquard_v2.json')
if not os.path.exists(SPLITS_PATH):
    !python tools/prepare_split.py --root {DATA_DIR} --out {SPLITS_PATH} --val-frac 0.1 --test-frac 0.1 --seed 0
else:
    print('split already exists, reusing:', SPLITS_PATH)

## 5. Optional — copy a previous run from an input dataset to resume

Kaggle notebook sessions are **9 h max** and **wipe `/kaggle/working/`** between runs. To survive multiple sessions:
1. Run the notebook once until `last.pth` exists in `/kaggle/working/runs/<RUN_NAME>/`.
2. Click **Save Version** → after it finishes, the working folder becomes downloadable as the notebook's output.
3. Create a new Kaggle Dataset from that output (or use the notebook output directly), and attach it as an input to the next run.
4. Set `RESUME_INPUT_DIR` in cell 0 to that input path.

In [ ]:
SAVE_DIR = f'/kaggle/working/runs/{RUN_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)

if RESUME_INPUT_DIR and os.path.isdir(RESUME_INPUT_DIR):
    print(f'copying previous run from {RESUME_INPUT_DIR} → {SAVE_DIR}')
    for fname in os.listdir(RESUME_INPUT_DIR):
        src = os.path.join(RESUME_INPUT_DIR, fname)
        dst = os.path.join(SAVE_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    print('done')
else:
    print('no previous run attached; starting fresh')

## 6. Train

In [ ]:
RESUME_PATH = os.path.join(SAVE_DIR, 'last.pth')
resume_arg = f'--resume {RESUME_PATH}' if os.path.exists(RESUME_PATH) else ''
cmd = (f'python tools/train.py --config configs/default.yaml '
       f'dataset.splits_path={SPLITS_PATH} '
       f'dataset.image_size={IMAGE_SIZE} '
       f'dataset.num_workers={NUM_WORKERS} '
       f'trainer.epochs={EPOCHS} '
       f'trainer.batch_size={BATCH_SIZE} '
       f'trainer.accum_steps={ACCUM_STEPS} '
       f'trainer.save_dir={SAVE_DIR} '
       f'{resume_arg}')
print(cmd)
!{cmd}

## 7. Plot training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(os.path.join(SAVE_DIR, 'metrics.csv'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if 'train_loss' in df.columns:
    axes[0].plot(df['epoch'], df['train_loss'], label='train_loss')
axes[0].set_title('train loss'); axes[0].legend(); axes[0].grid()
for col in ['val_miou_fg', 'val_dice_fg']:
    if col in df.columns:
        axes[1].plot(df['epoch'], df[col], label=col)
axes[1].set_title('val metrics'); axes[1].legend(); axes[1].grid()
plt.tight_layout(); plt.show()